In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn import set_config
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
from xgboost import XGBRegressor

In [3]:
import optuna
import mlflow
import dagshub

c:\Users\Jay Kanakia\Desktop\CampusX\Projects\uber-demand-forecasting\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# loading train and test dataset

# train data
train_df_path = r"..\data\processed\train.csv"
train_df = pd.read_csv(train_df_path)

# test data
test_df_path = r"..\data\processed\test.csv"
test_df = pd.read_csv(test_df_path)

In [5]:
train_df

,region,total_pickups,avg_pickups,day_of_week,month,lag_1,lag_2,lag_3,lag_4
0,0,333,324.0,4,1,345.0,326.0,302.0,197.0
1,0,329,326.0,4,1,333.0,345.0,326.0,302.0
2,0,313,321.0,4,1,329.0,333.0,345.0,326.0
3,0,271,301.0,4,1,313.0,329.0,333.0,345.0
4,0,258,283.0,4,1,271.0,313.0,329.0,333.0
...,...,...,...,...,...,...,...,...,...
172675,29,351,370.0,0,2,359.0,357.0,406.0,415.0
172676,29,248,321.0,0,2,351.0,359.0,357.0,406.0
172677,29,176,263.0,0,2,248.0,351.0,359.0,357.0
172678,29,184,231.0,0,2,176.0,248.0,351.0,359.0


In [6]:
# missing values in train_df

train_df.isnull().sum()

region           0
total_pickups    0
avg_pickups      0
day_of_week      0
month            0
lag_1            0
lag_2            0
lag_3            0
lag_4            0
dtype: int64

In [7]:
# missing values in test_df

test_df.isnull().sum()

region           0
total_pickups    0
avg_pickups      0
day_of_week      0
month            0
lag_1            0
lag_2            0
lag_3            0
lag_4            0
dtype: int64

In [8]:
# Splitting into X_train and y_train

X_train = train_df.drop(columns=['total_pickups'])
y_train =train_df[['total_pickups']]

In [9]:
# Splitting into X_test and y_test

X_test = test_df.drop(columns=['total_pickups'])
y_test =test_df[['total_pickups']]

In [10]:
# encoder

encoder = ColumnTransformer(
    [
        ('OHE',OneHotEncoder(drop='first',sparse_output=True),['region','day_of_week'])
    ],
    remainder='passthrough',verbose_feature_names_out=False
)

encoder

,transformers,"[('OHE', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,True


In [11]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [12]:
dagshub.init(repo_owner='jay-kanakia', repo_name='uber-demand-forecasting', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/jay-kanakia/uber-demand-forecasting.mlflow')

Accessing as jay-kanakia

Initialized MLflow to track repo "jay-kanakia/uber-demand-forecasting"

Repository jay-kanakia/uber-demand-forecasting initialized!

In [13]:
mlflow.set_experiment('Model : Selection')

<Experiment: artifact_location='mlflow-artifacts:/d96699b66897404787ec955910f704a4', creation_time=1777815059256, experiment_id='3', last_update_time=1777815059256, lifecycle_stage='active', name='Model : Selection', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [14]:
def objective(trial):

    # start the child run
    with mlflow.start_run(nested=True) as child:

        # model name search space
        model_list = ['LR','RF','GBR','XGBR']
        model_name = trial.suggest_categorical('model_name',model_list)

        if model_name == 'LR':
            model = LinearRegression()

        elif model_name == 'RF':
            n_estimators_rf = trial.suggest_int('n_estimators',10,100,step=10)
            max_depth_rf = trial.suggest_int('max_depth_rf',3,10)
            model = RandomForestRegressor(n_estimators=n_estimators_rf,max_depth=max_depth_rf,random_state=42,n_jobs=-1)

        elif model_name == 'GBR':
            n_estimators_gb = trial.suggest_int('n_estimators_gb',10,100,step=10)
            learning_rate_gb = trial.suggest_float('learning_rate_gb',1e-4,1e-1,log=True)
            model = GradientBoostingRegressor(n_estimators=n_estimators_gb,learning_rate=learning_rate_gb,random_state=42)

        elif model_name == 'XGBR':
            n_estimators_xgb = trial.suggest_int('n_estimators_xgb',10,100,step=10)
            learning_rate_xgb = trial.suggest_float('learning_rate_xgb',1e-4,1e-1,log=True)
            max_depth_xgb = trial.suggest_int('max_depth_xgb',3,10)
            model = XGBRegressor(n_estimators=n_estimators_xgb,learning_rate=learning_rate_xgb,max_depth=max_depth_xgb)

        # log the model name
        mlflow.log_param('model_name',model_name)

        # log the model parameters
        mlflow.log_params(model.get_params())

        # fit the data
        model.fit(X_train_encoded,y_train)

        # get the prediction
        y_pred = model.predict(X_test_encoded)

        # calculate the loss
        loss = mean_absolute_percentage_error(y_test,y_pred)

        # log the metric
        mlflow.log_metric('MAPE',loss)

        return loss



In [15]:
with mlflow.start_run(run_name='Best Model'):

    sampler = optuna.samplers.TPESampler(seed=42)

    study = optuna.create_study(
        study_name='Model Selection',
        direction='minimize',
        sampler=sampler
    )

    study.optimize(func=objective, n_trials=100, n_jobs=1)

    mlflow.log_params(study.best_params)
    mlflow.log_metric('best_mape', study.best_value)

    mlflow.log_param("best_model", study.best_params.get("model_name"))

[I 2026-05-03 19:49:54,512] A new study created in memory with name: Model Selection
c:\Users\Jay Kanakia\Desktop\CampusX\Projects\uber-demand-forecasting\myenv\lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


🏃 View run carefree-moose-380 at: https://dagshub.com/jay-kanakia/uber-demand-forecasting.mlflow/#/experiments/3/runs/333048b65fb6419bb3cf803dd5b2661e
🧪 View experiment at: https://dagshub.com/jay-kanakia/uber-demand-forecasting.mlflow/#/experiments/3


[I 2026-05-03 19:49:58,270] Trial 0 finished with value: 0.3520774173849667 and parameters: {'model_name': 'RF', 'n_estimators': 20, 'max_depth_rf': 4}. Best is trial 0 with value: 0.3520774173849667.
c:\Users\Jay Kanakia\Desktop\CampusX\Projects\uber-demand-forecasting\myenv\lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


🏃 View run enthused-gnu-512 at: https://dagshub.com/jay-kanakia/uber-demand-forecasting.mlflow/#/experiments/3/runs/7176f52b057e40d8a62c0e44b4c07d6a
🧪 View experiment at: https://dagshub.com/jay-kanakia/uber-demand-forecasting.mlflow/#/experiments/3


[I 2026-05-03 19:50:05,838] Trial 1 finished with value: 0.13000572299916016 and parameters: {'model_name': 'RF', 'n_estimators': 10, 'max_depth_rf': 10}. Best is trial 1 with value: 0.13000572299916016.


...

🏃 View run orderly-chimp-185 at: https://dagshub.com/jay-kanakia/uber-demand-forecasting.mlflow/#/experiments/3/runs/b87179a95a91417591e26696e72c9c14
🧪 View experiment at: https://dagshub.com/jay-kanakia/uber-demand-forecasting.mlflow/#/experiments/3


[I 2026-05-03 20:01:43,415] Trial 98 finished with value: 0.09111598757158389 and parameters: {'model_name': 'LR'}. Best is trial 2 with value: 0.09111598757158389.


🏃 View run marvelous-shoat-832 at: https://dagshub.com/jay-kanakia/uber-demand-forecasting.mlflow/#/experiments/3/runs/f8efba1fb2324408b26b5447d014bcc8
🧪 View experiment at: https://dagshub.com/jay-kanakia/uber-demand-forecasting.mlflow/#/experiments/3


[I 2026-05-03 20:01:49,447] Trial 99 finished with value: 0.09111598757158389 and parameters: {'model_name': 'LR'}. Best is trial 2 with value: 0.09111598757158389.


🏃 View run Best Model at: https://dagshub.com/jay-kanakia/uber-demand-forecasting.mlflow/#/experiments/3/runs/d8a34df6a15a4c8280e33c1f1d98b246
🧪 View experiment at: https://dagshub.com/jay-kanakia/uber-demand-forecasting.mlflow/#/experiments/3


In [16]:
# best model

study.best_params

{'model_name': 'LR'}

In [17]:
# best trial

study.best_trial

FrozenTrial(number=2, state=<TrialState.COMPLETE: 1>, values=[0.09111598757158389], datetime_start=datetime.datetime(2026, 5, 3, 19, 50, 5, 844235), datetime_complete=datetime.datetime(2026, 5, 3, 19, 50, 8, 360845), params={'model_name': 'LR'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'model_name': CategoricalDistribution(choices=('LR', 'RF', 'GBR', 'XGBR'))}, trial_id=2, value=None)

In [18]:
# best value

study.best_value

0.09111598757158389

In [19]:
# model value counts

study.trials_dataframe()['params_model_name'].value_counts()

params_model_name
LR      71
RF      11
XGBR     9
GBR      9
Name: count, dtype: int64

### Visualization

In [20]:
import importlib
import optuna.visualization._plotly_imports as p

importlib.reload(p)

<module 'optuna.visualization._plotly_imports' from 'c:\\Users\\Jay Kanakia\\Desktop\\CampusX\\Projects\\uber-demand-forecasting\\myenv\\lib\\site-packages\\optuna\\visualization\\_plotly_imports.py'>

In [21]:
optuna.visualization.plot_optimization_history(study)

In [24]:
optuna.visualization.plot_parallel_coordinate(study,params=['model_name'])

In [25]:
# train the LR model on best settings

# obeject creation
lr = LinearRegression()

# fit the train data
lr.fit(X_train_encoded,y_train)

# get the prediction
y_pred_train = lr.predict(X_train_encoded)
y_pred_test = lr.predict(X_test_encoded)

# error calculation
train_MAPE = mean_absolute_percentage_error(y_train,y_pred_train)
test_MAPE = mean_absolute_percentage_error(y_test,y_pred_test)

print("The training error is ", train_MAPE)
print("The test error is ", test_MAPE)

The training error is  0.0961393564766014
The test error is  0.09111598757158389


In [26]:
lr.coef_

array([[ 1.96330288,  3.07601254,  3.42756714, -0.89429328,  4.04700181,
         4.03941692,  1.3447927 ,  0.16680284,  4.38501154,  1.90072567,
         2.89975767,  4.4128348 ,  0.50446494,  2.09439617,  2.63520781,
         4.38872313,  1.18317457,  2.82438852,  4.07110361,  0.43253806,
         4.40215482,  1.6727597 ,  2.10007561,  1.36936139,  1.14734485,
        -0.17110611,  2.26066948,  4.17623493, -0.04707795, -0.13473219,
        -0.31041868, -0.39810599, -0.56782607, -0.45179986, -0.37703018,
         2.49449438, -0.26900214, -0.54448272, -0.36403731, -0.25156   ,
        -0.31553451]])

In [27]:
from sklearn.linear_model import Ridge

def tune_ridge(trial):
    
    # hyperparameter space
    alpha = trial.suggest_float('alpha',10,100)

    # model object
    ridge = Ridge(alpha=alpha,random_state=42)

    # train the model
    ridge.fit(X_train_encoded,y_train)

    # get prediction
    y_pred = ridge.predict(X_test_encoded)

    # error calculation
    loss  = mean_absolute_percentage_error(y_test,y_pred)

    return loss



In [29]:
# create study 

study = optuna.create_study(study_name='tune_model',direction='minimize')

study

[I 2026-05-03 20:14:42,762] A new study created in memory with name: tune_model


In [30]:
# optimize

study.optimize(func=tune_ridge,n_trials=100,show_progress_bar=True)

Best trial: 1. Best value: 0.104722:   2%|▏         | 2/100 [00:00<00:13,  7.39it/s]

[I 2026-05-03 20:15:23,501] Trial 0 finished with value: 0.10472560975467839 and parameters: {'alpha': 12.248969774116329}. Best is trial 0 with value: 0.10472560975467839.
[I 2026-05-03 20:15:23,635] Trial 1 finished with value: 0.10472192857995766 and parameters: {'alpha': 17.115154424876255}. Best is trial 1 with value: 0.10472192857995766.


Best trial: 1. Best value: 0.104722:   4%|▍         | 4/100 [00:00<00:12,  7.45it/s]

[I 2026-05-03 20:15:23,752] Trial 2 finished with value: 0.10472818324862308 and parameters: {'alpha': 57.52757406846436}. Best is trial 1 with value: 0.10472192857995766.
[I 2026-05-03 20:15:23,888] Trial 3 finished with value: 0.10472481599541458 and parameters: {'alpha': 49.6701320607783}. Best is trial 1 with value: 0.10472192857995766.


Best trial: 1. Best value: 0.104722:   6%|▌         | 6/100 [00:00<00:11,  8.07it/s]

...

[I 2026-05-03 20:15:31,871] Trial 94 finished with value: 0.10472685524992302 and parameters: {'alpha': 59.63521135745485}. Best is trial 47 with value: 0.10471834126772704.
[I 2026-05-03 20:15:31,949] Trial 95 finished with value: 0.10472683004184055 and parameters: {'alpha': 21.1794291745804}. Best is trial 47 with value: 0.10471834126772704.
[I 2026-05-03 20:15:32,031] Trial 96 finished with value: 0.10473581409363525 and parameters: {'alpha': 28.921503766186657}. Best is trial 47 with value: 0.10471834126772704.


Best trial: 47. Best value: 0.104718:  98%|█████████▊| 98/100 [00:08<00:00, 12.08it/s]

[I 2026-05-03 20:15:32,115] Trial 97 finished with value: 0.10473542143506641 and parameters: {'alpha': 10.040527099661668}. Best is trial 47 with value: 0.10471834126772704.
[I 2026-05-03 20:15:32,216] Trial 98 finished with value: 0.10473549592942122 and parameters: {'alpha': 16.43821425767708}. Best is trial 47 with value: 0.10471834126772704.


Best trial: 47. Best value: 0.104718: 100%|██████████| 100/100 [00:08<00:00, 11.20it/s]

[I 2026-05-03 20:15:32,294] Trial 99 finished with value: 0.1047351806914354 and parameters: {'alpha': 12.273118724551294}. Best is trial 47 with value: 0.10471834126772704.


In [31]:
study.best_params

{'alpha': 21.531598074482186}

In [32]:
study.best_trial

FrozenTrial(number=47, state=<TrialState.COMPLETE: 1>, values=[0.10471834126772704], datetime_start=datetime.datetime(2026, 5, 3, 20, 15, 27, 953846), datetime_complete=datetime.datetime(2026, 5, 3, 20, 15, 28, 24919), params={'alpha': 21.531598074482186}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'alpha': FloatDistribution(high=100.0, log=False, low=10.0, step=None)}, trial_id=47, value=None)

In [33]:
study.best_value

0.10471834126772704